# Wan 2.2 videomotor til Agentfirm

Denne notebook starter **Wan 2.2 (TI2V-5B)** på en Colab-GPU og giver den en sikker
internetadresse, som agenterne i Agentfirm kan sende video-opgaver til.

**Krav:** Colab Pro eller Pay-as-you-go. Vælg *Runtime → Change runtime type* →
**A100 GPU** (hurtigst) eller **L4 GPU** + **High-RAM**. Den gratis T4 har for lidt
hukommelse til modellen.

**Sådan gør du:** *Runtime → Run all*. Første gang tager det 10–20 minutter at hente
modellen. Til sidst printes en **adresse** og en **nøgle**, som du sætter ind i
Agentfirm (se sidste celle).

Motoren kører kun, mens denne fane er åben og Colab-sessionen lever.

In [ ]:
# 1. Tjek GPU og hukommelse
import subprocess, psutil
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "INGEN GPU – vælg en GPU-runtime")
ram_gb = psutil.virtual_memory().total / 1e9
print(f"System-RAM: {ram_gb:.0f} GB")
if ram_gb < 40:
    print("ADVARSEL: For lidt RAM. Vælg A100, eller L4 med High-RAM.")

In [ ]:
# 2. Installer pakker
!pip -q install "diffusers>=0.35.1" "transformers>=4.49" accelerate ftfy imageio imageio-ffmpeg fastapi "uvicorn[standard]"

In [ ]:
# 3. Hent API-serveren fra GitHub
BRANCH = "HEAD"  # skift til en branch, hvis du tester en ny version
!wget -q -O /content/wan_server.py https://raw.githubusercontent.com/MagnoraMarketing/MagnoraMarketing-Agentfirm/{BRANCH}/tools/wan-colab/wan_server.py
print("OK")

In [ ]:
# 4. Nøgle til API'en.
# Gem gerne en fast nøgle som Colab-secret med navnet WAN_API_TOKEN (nøgle-ikonet i venstre side),
# så du ikke skal opdatere den i Agentfirm hver gang.
import os, secrets
token = None
try:
    from google.colab import userdata
    token = userdata.get("WAN_API_TOKEN")
except Exception:
    pass
if not token:
    token = secrets.token_urlsafe(32)
    print("Ingen gemt nøgle fundet – lavede en ny for denne session.")
os.environ["WAN_API_TOKEN"] = token
os.environ["WAN_OUTPUT_DIR"] = "/content/wan_outputs"

In [ ]:
# 5. Indlæs Wan 2.2 og start API'en (første gang: 10–20 minutter)
import sys, threading, time, uvicorn
sys.path.insert(0, "/content")
import wan_server
app = wan_server.create_app()
server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(3)
print("API kører lokalt på port 8000")

In [ ]:
# 6. Åbn en sikker tunnel (Cloudflare, gratis, ingen konto)
import re, subprocess, time
!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public_url = None
deadline = time.time() + 60
while time.time() < deadline and not public_url:
    line = tunnel.stdout.readline()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        public_url = m.group(0)
print("Tunnel:", public_url or "FEJL – kør cellen igen")

In [ ]:
# 7. Test og vis det, du skal sætte ind i Agentfirm
import requests
print(requests.get(f"{public_url}/health", timeout=30).json())
print()
print("Sæt disse to værdier ind i Agentfirm (Company Settings → Secrets):")
print("  WAN_API_URL   =", public_url)
print("  WAN_API_TOKEN =", token)
print()
print("Bind dem som environment variables på de agenter, der skal lave video.")
print("Adressen skifter hver gang notebooken startes. Opdater WAN_API_URL, når det sker.")

## Lav en testvideo herfra (valgfrit)

In [ ]:
import time, requests
H = {"Authorization": f"Bearer {token}"}
job = requests.post(f"{public_url}/jobs", headers=H, json={
    "prompt": "A slow cinematic drone shot over a Danish coastline at sunset, waves, warm light",
    "orientation": "landscape", "seconds": 3, "steps": 25,
}).json()
print("Job:", job["id"])
while True:
    s = requests.get(f"{public_url}/jobs/{job['id']}", headers=H).json()
    print(s["status"]); 
    if s["status"] in ("succeeded", "failed"): break
    time.sleep(15)
if s["status"] == "succeeded":
    from IPython.display import Video
    path = f"/content/wan_outputs/{job['id']}.mp4"
    display(Video(path, embed=True, width=640))
else:
    print(s["error"])

## Hold motoren i gang
Lad fanen stå åben. Cellen nedenfor viser status hvert minut, så du kan se, at den lever.

In [ ]:
while True:
    try:
        print(time.strftime("%H:%M"), requests.get(f"{public_url}/health", timeout=30).json())
    except Exception as e:
        print(time.strftime("%H:%M"), "fejl:", e)
    time.sleep(60)